# Сбор признаков МО: Росстат и география

Собираем статичные признаки муниципальных образований, чтобы дополнить ими траты СберИндекса. В эксперименте 1 (`mo_clustering.ipynb`) кластеры по одним тратам получились «по размеру города»; эти признаки должны описать профиль МО: отраслевую структуру, демографию, статус, положение в пространстве.

**Что собираем**

| Блок | Источник | Файл-лоадер |
|---|---|---|
| Население, занятость, зарплата, отгрузка по видам деятельности, инвестиции, инфраструктура | БДМО Росстата (каталог tochno.st) | `datasets/bdmo.py` |
| Площадь, расстояние до крупных городов, пригород, соседи, доступность рынка | границы МО + население | `datasets/geo.py` |

**Структура проекта**

```
data/                     файлы, которые вы скачиваете сами
  bdmo_catalog.csv        каталог показателей БДМО (таблица tochno.st)
  bdmo/*.zip              архивы показателей БДМО (качает лоадер)
  Municipalities_Russia_2025.geojson   границы МО с кодами ОКТМО
datasets/                 лоадеры (этот ноутбук их только вызывает)
results/mo_features.csv   итоговая таблица
```

**Ключ соединения - ОКТМО.** Все лоадеры возвращают таблицу, индексированную кодом ОКТМО из 8 знаков. По названиям МО ничего не соединяется: в данных встречаются одинаковые названия (районы Москвы) и разные формы одного топонима.

**Результат - статичная таблица**: одна строка на МО. Траты остаются отдельным временным рядом; как их соединить, показано в конце.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))      # чтобы импортировался пакет datasets из этой папки
from datasets import bdmo, boundaries, common, geo

DATA = Path('data')
OUT = Path('results'); OUT.mkdir(exist_ok=True)

COSTS_PATH = DATA / 'potrebitelskie-beznalicnye-rashody-na-urovne-munizipalnyh-obrazovanij_ru_1764079373653.parquet'
BDMO_DIR = DATA / 'bdmo'                        # сюда скачиваются архивы показателей
BOUNDARIES_PATH = DATA / 'Municipalities_Russia_2025.geojson'
SOURCE_CRS = 3857                               # система координат файла границ

YEARS = [2022, 2023, 2024, 2025]                # из каких лет брать показатели Росстата

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

## 1. Базовый список МО и границы

Базовый список задаёт датасет трат: он определяет, какие МО участвуют в исследовании. Но в нём МО заданы только названием, а ключ для соединения со всеми внешними источниками - ОКТМО. Код и геометрию даёт файл границ.

Связываются они по **сокращённому названию**: из полного названия берутся слова с заглавной буквы, то есть топоним без типа МО («городской округ город Орёл» → «Орёл»).

**Неоднозначные названия отбрасываются.** «Кировский район» есть в нескольких регионах, и по названию из трат определить, какой именно имеется в виду, невозможно. Соединение по такому ключу дало бы декартово произведение и молча размножило строки временного ряда, поэтому такие МО исключаются из анализа, а не угадываются. Сколько их, показывает сводка.

In [2]:
costs = pd.read_parquet(COSTS_PATH)
mo_boundaries = boundaries.load(BOUNDARIES_PATH, crs=SOURCE_CRS)
display(mo_boundaries.head())

Один ОКТМО у нескольких объектов: 9 строк, они объединены
Границы: 2666 МО


,oktmo,name,short_name,geometry
0,00000000,Гуляйпольский муниципальный округ,Гуляйпольский,"MULTIPOLYGON (((34.73308 47.11084, 34.72448 47..."
1,01512000,Залесовский муниципальный округ,Залесовский,"POLYGON ((85.38619 54.21921, 85.28016 54.17862..."
2,01514000,Муниципальный округ Змеиногорский район,Змеиногорский,"POLYGON ((82.54031 51.17237, 82.5498 51.18325,..."
3,01539000,Муниципальный округ город Славгород,Славгород,"POLYGON ((79.63868 53.28401, 79.64014 53.27952..."
4,01541000,Муниципальный округ Суетский район,Суетский,"POLYGON ((79.81884 53.38451, 79.87649 53.37583..."


In [3]:
mapping = boundaries.match(mo_boundaries, costs['mo'])

costs = costs.drop(columns=['oktmo'], errors='ignore').merge(mapping[['mo', 'oktmo']], on='mo', how='inner')
base = common.load_base(costs)
print(f'\nВ анализе: {len(base)} МО')
display(base.head())

МО в тратах: 2118
  сопоставлено: 1855
  отброшено как неоднозначные: 226 (сокращённых названий: 129)
  нет в границах: 37

Примеры отброшенных:
                                                                              mo      short_name
 внутригородская территория города федерального значения муниципальный округ № 7                
внутригородская территория города федерального значения муниципальный округ № 54                
внутригородская территория города федерального значения муниципальный округ № 65                
внутригородская территория города федерального значения муниципальный округ № 78                
внутригородская территория города федерального значения муниципальный округ № 15                
внутригородская территория города федерального значения муниципальный округ № 72                
внутригородская территория города федерального значения муниципальный округ № 21                
                              Азовский немецкий национальный муниципальный райо

,mo,oktmo,oktmo_region,mo_norm
0,Бай-Тайгинский муниципальный район,93605000,93,бай тайгинский
1,Красночетайский муниципальный округ,97526000,97,красночетайский
2,Тербунский муниципальный район,42645000,42,тербунский
3,Гаврилов-Ямский муниципальный район,78612000,78,гаврилов ямский
4,Кичменгско-Городецкий муниципальный округ,19530000,19,кичменгско


## 2. Росстат: БДМО

Данные опубликованы по одному архиву на показатель, всего 603 показателя, каталог со ссылками - отдельная таблица. Показатели заданы **кодами**, а не названиями: названия в БДМО не уникальны. Например, «Оборот розничной торговли (без субъектов малого предпринимательства)» - это и 8201003 (2009–2016, по ОКВЭД), и 8401003 (2010–2026, по ОКВЭД-2). Отбор по названию дал бы либо обрыв ряда, либо случайный выбор из двух.

Коды собраны в `bdmo.INDICATOR`: для каждого указаны разрез и покрытие (доля МО, у которых есть значение). Показатель с покрытием 20% в кластеризации бесполезен, поэтому в реестр вошли только показатели с покрытием от 80%.

In [4]:
# Каталог: скачанная копия таблицы tochno.st.
# Можно и напрямую: catalog = bdmo.load_catalog()  - тогда читается гугл-таблица
catalog = bdmo.load_catalog(DATA / 'bdmo_catalog.csv')
display(bdmo.registry().head(20))

Каталог: 603 показателей, 33 разделов, годы покрытия 2005–2026


,код,разрез,покрытие,показатель
имя,,,,
население,8112027,mest,0.99,Оценка численности населения на 1 января
население_среднегод,8112013,NaN,0.96,Среднегодовая численность постоянного населения
население_возраст,8112014,"grup_2, vozr",0.94,Численность по полу и возрасту на 1 января
коэф_рождаемости,8112007,NaN,0.94,Общий коэффициент рождаемости (ряд до 2023)
коэф_смертности,8112008,NaN,0.94,Общий коэффициент смертности (ряд до 2023)
коэф_естприроста,8112009,NaN,0.94,Общий коэффициент естественного прироста (ряд ...
миграционный_прирост,8112023,"migr, grup_2, vozr",0.97,"Миграционный прирост; в 2024 покрытие 1%, брат..."
работники,8423005,okved2,0.99,Среднесписочная численность работников (без ма...
фот,8423006,okved2,0.99,Фонд заработной платы (без малого бизнеса)


### Что берём

`bdmo.CORE` - стартовый набор: высокое покрытие, нет разрезов, многие признаки уже нормированы на человека или выражены в процентах. Отдельно собирается отраслевая структура по ОКВЭД (следующий шаг) - ради неё и нужны показатели с разрезами.

Остальные показатели реестра добавляются по имени: `bdmo.codes(bdmo.CORE + ['выручка', 'сельхоз_продукция'])`.

In [5]:
INDICATORS = bdmo.codes(bdmo.CORE)
display(bdmo.registry(bdmo.CORE))

paths = bdmo.download(catalog, INDICATORS.values(), folder=BDMO_DIR)

,код,разрез,покрытие,показатель
имя,,,,
население_среднегод,8112013,None,0.96,Среднегодовая численность постоянного населения
организации,9010002,None,0.99,Количество организаций по госрегистрации (с 2024)
ип,9010003,None,0.99,Количество ИП по госрегистрации (с 2024)
зарплата_крупные,8213002,None,0.94,"Зарплата на крупных и средних предприятиях, бе..."
инвестиции_на_душу,8109003,None,0.96,Инвестиции (кроме бюджетных) на одного человека
продтовары_объем,8801004,None,0.94,Объём продовольственных товаров в границах МО
средства_размещения,8060001,None,0.89,Число коллективных средств размещения (туризм)
площадь_земель,8006001,None,0.99,"Общая площадь земель МО, га"
улицы,8006007,None,0.94,"Общая протяжённость улиц, проездов, набережных"


8112013: уже скачан (12.6 МБ)
9010002: уже скачан (12.2 МБ)
9010003: уже скачан (13.0 МБ)
8213002: уже скачан (6.2 МБ)
8109003: уже скачан (3.2 МБ)
8801004: уже скачан (1.6 МБ)
8060001: уже скачан (4.7 МБ)
8006001: уже скачан (78.1 МБ)
8006007: уже скачан (62.7 МБ)
8106005: уже скачан (6.9 МБ)
8211001: уже скачан (3.6 МБ)
8215001: уже скачан (3.0 МБ)
9010004: уже скачан (0.8 МБ)
8018000: уже скачан (15.6 МБ)
8014006: уже скачан (2.7 МБ)
8313015: уже скачан (3.1 МБ)
8313004: уже скачан (2.9 МБ)
8112007: уже скачан (16.3 МБ)
8112008: уже скачан (16.1 МБ)
8112009: уже скачан (16.6 МБ)


In [6]:
rosstat = bdmo.load(catalog, INDICATORS, oktmo=base['oktmo'].dropna().tolist(),
                    folder=BDMO_DIR, years=YEARS, agg='last')
print(f'\nПоказателей собрано: {sum(1 for c in rosstat.columns if not c.endswith("_year"))}, МО: {len(rosstat)}')
display(rosstat.head())

население_среднегод (8112013): 4936 повторов на пару «МО + год», взято среднее. Проверьте разрезы: bdmo.breakdowns(bdmo.read_indicator(path))
население_среднегод (8112013): 1531 МО — Среднегодовая численность постоянного населения
организации (9010002): 19510 повторов на пару «МО + год», взято среднее. Проверьте разрезы: bdmo.breakdowns(bdmo.read_indicator(path))
организации (9010002): 1572 МО — Количество организаций по данным государственной регистрации
ип (9010003): 19522 повторов на пару «МО + год», взято среднее. Проверьте разрезы: bdmo.breakdowns(bdmo.read_indicator(path))
ип (9010003): 1572 МО — Количество индивидуальных предпринимателей по данным государ
зарплата_крупные (8213002): 5891 повторов на пару «МО + год», взято среднее. Проверьте разрезы: bdmo.breakdowns(bdmo.read_indicator(path))
зарплата_крупные (8213002): 1488 МО — Среднемесячная номинальная начисленная заработная плата рабо
инвестиции_на_душу (8109003): 4622 повторов на пару «МО + год», взято среднее. Проверьте ра

,население_среднегод,население_среднегод_year,организации,организации_year,ип,ип_year,зарплата_крупные,зарплата_крупные_year,инвестиции_на_душу,инвестиции_на_душу_year,продтовары_объем,продтовары_объем_year,средства_размещения,средства_размещения_year,площадь_земель,площадь_земель_year,улицы,улицы_year,дороги_плохие_доля,дороги_плохие_доля_year,жилье_на_жителя,жилье_на_жителя_year,жилье_ввод_на_жителя,жилье_ввод_на_жителя_year,жилье_благоустр,жилье_благоустр_year,лпу,лпу_year,детсады_охват,детсады_охват_year,доля_своих_доходов,доля_своих_доходов_year,расходы_на_омсу,расходы_на_омсу_year,коэф_рождаемости,коэф_рождаемости_year,коэф_смертности,коэф_смертности_year,коэф_естприроста,коэф_естприроста_year
oktmo,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
01512000,11162.0,2024.0,70.00,2025,220.25,2025,51062.3,2025.0,1341.0,2024.0,827055.0,2025.0,5.0,2025.0,327456.0,2024,354.8,2024.0,10.3,2025.0,29.70,2024.0,0.088,2024.0,29.78,2025.0,12.0,2024.0,55.6,2024.0,38.0,2025.0,4722.0,2024.0,9.0,2023.0,17.3,2023.0,-8.3,2023.0
01514000,15746.0,2024.0,127.75,2025,302.50,2025,91151.7,2025.0,340013.0,2024.0,1481038.0,2025.0,18.0,2025.0,280161.0,2024,277.7,2024.0,75.4,2025.0,31.66,2024.0,0.138,2024.0,150.68,2025.0,17.0,2024.0,65.7,2024.0,51.0,2025.0,5336.0,2024.0,9.0,2023.0,18.1,2023.0,-9.1,2023.0
01539000,36654.0,2024.0,250.75,2025,868.25,2025,53475.0,2025.0,8403.0,2024.0,3468275.0,2025.0,5.0,2025.0,213661.0,2024,292.3,2024.0,84.7,2025.0,26.31,2024.0,0.099,2024.0,258.11,2025.0,28.0,2024.0,48.8,2024.0,48.0,2025.0,2192.0,2024.0,7.6,2023.0,13.4,2023.0,-5.8,2023.0
01541000,3208.0,2024.0,25.25,2025,49.00,2025,64991.5,2025.0,129663.0,2024.0,175470.0,2025.0,NaN,NaN,110818.0,2024,81.4,2024.0,68.3,2025.0,46.80,2024.0,0.028,2024.0,15.80,2025.0,8.0,2024.0,79.0,2024.0,35.0,2025.0,9712.0,2024.0,7.2,2023.0,21.7,2023.0,-14.5,2023.0
01558000,9194.0,2024.0,88.00,2025,235.00,2025,44972.4,2025.0,3244.0,2024.0,505825.0,2025.0,16.0,2025.0,688140.0,2024,242.3,2024.0,94.4,2025.0,29.55,2024.0,0.166,2024.0,45.00,2025.0,16.0,2024.0,54.2,2024.0,22.0,2025.0,5371.0,2024.0,9.3,2023.0,18.2,2023.0,-8.9,2023.0


### Отраслевая структура по ОКВЭД

Главный признак для исходной задачи. По долям отгрузки видно, добывающее МО, обрабатывающее или сервисное. Абсолютные значения отражали бы размер, как в эксперименте с кластеризацией по временным рядам.

Показатель 8401011 «Отгружено товаров собственного производства» в разрезе `okved2` даёт по колонке на раздел ОКВЭД. Три момента, без которых доли получаются неверными:

* **Разрез смешивает уровни классификации.** Рядом с разделами ОКВЭД в нём лежат вложенные в них группы («Торговля розничная моторным топливом»). Складывать их вместе - считать одно и то же дважды, поэтому берём только разделы по шаблону `^Раздел`.
* **Пропуск означает ноль, а не «неизвестно».** Если раздела нет в списке по МО, его в МО нет или он скрыт по конфиденциальности. Для доли это ноль. Иначе у каждого раздела своё покрытие, и отбор по полноте оставит только вездесущие бюджетные отрасли.
* **Остаток выносится в колонку «нераскрыто»** - какая часть «Всего» не объяснена разделами. Высокое значение означает, что экономика держится на одном-двух работодателях, чьи данные закрыты: прямая подпись моногорода.

In [7]:
# Сначала посмотрим, какие категории вообще есть в разрезе
bdmo.download(catalog, bdmo.codes(bdmo.BY_OKVED).values(), folder=BDMO_DIR)
bdmo.breakdowns(bdmo.read_indicator(BDMO_DIR / f"bdmo_{bdmo.INDICATOR['отгружено'][0]}.zip"), limit=25)

8401011: уже скачан (27.7 МБ)
8423005: уже скачан (328.3 МБ)
7000005: уже скачан (171.4 МБ)
8423006: уже скачан (351.6 МБ)
okved2: 32 значений
    Всего по обследуемым видам экономической деятельности
    Раздел D Обеспечение электрической энергией, газом и паром, кондиционирование воздуха
    Раздел P Образование
    Раздел G Торговля оптовая и розничная, ремонт автотранспортных средств и мотоциклов
    Раздел А Сельское, лесное хозяйство, охота, рыболовство и рыбоводство
    Раздел M Деятельность профессиональная, научная и техническая
    Раздел F Строительство
    Раздел C Обрабатывающие производства
    Раздел L Деятельность по операциям с недвижимым имуществом
    Раздел J Деятельность в области информации и связи
    Раздел Н Транспортировка и хранение
    Раздел I Деятельность гостиниц и предприятий общественного питания
    Раздел Q Деятельность в области здравоохранения и социальных услуг
    Раздел R Деятельность в области культуры, спорта, организации досуга и развлечений
 

In [8]:
okved = {}
for name in ['отгружено', 'работники']:          # структура отгрузки и структура занятости
    okved[name] = bdmo.load_breakdown(
        catalog, bdmo.INDICATOR[name][0], oktmo=base['oktmo'].dropna().tolist(),
        column='okved2', folder=BDMO_DIR, as_shares=True, min_share=0.01,
        pattern=r'^Раздел',        # только разделы ОКВЭД: в разрезе есть и вложенные в них группы
        prefix=f'{name}_доля_')

okved_features = pd.concat(okved.values(), axis=1)
print(f'Колонок отраслевой структуры: {okved_features.shape[1]}')
display(okved_features.describe().T[['count', 'mean', '50%', 'max']])

8401011: МО с данными по годам — 2017: 1518, 2018: 1536, 2019: 1566, 2020: 1569, 2021: 1568, 2022: 1570, 2023: 1570, 2024: 1570, 2025: 1569, 2026: 270
  выбран 2025 год
8401011, разрез «okved2», 2025 год: категорий 18, оставлено 16 (доля выше 1% хотя бы у 5% МО)
  доля нераскрытого: медиана 0.31, выше 0.5 у 41% МО
8423005: МО с данными по годам — 2017: 1550, 2018: 1572, 2019: 1572, 2020: 1572, 2021: 1570, 2022: 1572, 2023: 1572, 2024: 1572, 2025: 1571, 2026: 1135
  выбран 2025 год
8423005, разрез «okved2», 2025 год: категорий 19, оставлено 18 (доля выше 1% хотя бы у 5% МО)
  доля нераскрытого: медиана 0.06, выше 0.5 у 2% МО
Колонок отраслевой структуры: 36


,count,mean,50%,max
отгружено_доля_Раздел C Обрабатывающие производства,1525.0,0.196077,0.000000,0.979213
"отгружено_доля_Раздел D Обеспечение электрической энергией, газом и паром, кондиционирование воздуха",1525.0,0.049510,0.000000,0.937392
отгружено_доля_Раздел F Строительство,1525.0,0.020378,0.000000,0.956115
"отгружено_доля_Раздел G Торговля оптовая и розничная, ремонт автотранспортных средств и мотоциклов",1525.0,0.012693,0.001393,0.717739
отгружено_доля_Раздел I Деятельность гостиниц и предприятий общественного питания,1525.0,0.002991,0.000000,0.435244
отгружено_доля_Раздел J Деятельность в области информации и связи,1525.0,0.016214,0.000000,0.750996
отгружено_доля_Раздел L Деятельность по операциям с недвижимым имуществом,1525.0,0.008755,0.000000,0.430457
"отгружено_доля_Раздел M Деятельность профессиональная, научная и техническая",1525.0,0.016245,0.000000,0.810438
отгружено_доля_Раздел N Деятельность административная и сопутствующие дополнительные услуги,1525.0,0.005124,0.000000,0.617102
отгружено_доля_Раздел P Образование,1525.0,0.018037,0.004642,0.666859


### Производные признаки

Абсолютные значения зависят в основном от размера МО, поэтому переводим их в подушевые. Доли по ОКВЭД уже посчитаны выше, а часть показателей из `CORE` изначально нормирована (инвестиции на душу, жильё на жителя, проценты).

In [9]:
def per_capita(table, columns, population='население_среднегод', suffix='_на_1000'):
    # Абсолютный показатель → на 1000 жителей
    out = pd.DataFrame(index=table.index)
    if population not in table:
        print(f'Нет колонки «{population}», подушевые признаки пропущены')
        return out
    pop = table[population].replace(0, np.nan)
    for column in columns:
        if column in table:
            out[column + suffix] = table[column] / pop * 1000
    return out


ABSOLUTE = ['организации', 'ип', 'лпу', 'средства_размещения', 'продтовары_объем', 'улицы']
derived = per_capita(rosstat, ABSOLUTE)

# Плотность населения: площадь земель в гектарах
if {'население_среднегод', 'площадь_земель'} <= set(rosstat.columns):
    derived['плотность_на_км2'] = rosstat['население_среднегод'] / (rosstat['площадь_земель'] / 100)

keep = [c for c in rosstat.columns if not c.endswith('_year')]
rosstat_features = pd.concat([rosstat[keep], derived, okved_features], axis=1)
print(f'Признаков из Росстата: {rosstat_features.shape[1]}')
display(rosstat_features.describe().T[['count', 'mean', '50%', 'max']])

Признаков из Росстата: 63


,count,mean,50%,max
население_среднегод,1531.0,62459.784237,23488.000000,1.635559e+06
организации,1572.0,1516.925785,260.625000,7.516350e+04
ип,1572.0,1957.638650,603.500000,7.286900e+04
зарплата_крупные,1488.0,84751.324496,69280.300000,3.878875e+05
инвестиции_на_душу,1561.0,320158.245159,33281.200000,8.009993e+07
...,...,...,...,...
"работники_доля_Раздел А Сельское, лесное хозяйство, охота, рыболовство и рыбоводство",1569.0,0.044291,0.000000,6.194379e-01
работники_доля_Раздел В Добыча полезных ископаемых,1569.0,0.023870,0.000000,7.384587e-01
"работники_доля_Раздел Е Водоснабжение, водоотведение, организация сбора и утилизация отходов, деятельность по ликвидации загрязнений",1569.0,0.005737,0.000000,1.966263e-01
работники_доля_Раздел Н Транспортировка и хранение,1569.0,0.061880,0.038282,6.353603e-01


### Из какого года значения

У Росстата лаг, и у разных показателей он разный. Колонки `*_year` показывают, из какого года взято каждое значение. Если у показателя разброс лет большой, признак смешивает разные годы по разным МО - такой лучше не брать.

In [10]:
year_cols = [c for c in rosstat.columns if c.endswith('_year')]
if year_cols:
    summary = pd.DataFrame({
        'год (медиана)': rosstat[year_cols].median(),
        'разных лет': rosstat[year_cols].nunique(),
        'МО со значением': rosstat[year_cols].notna().sum(),
    })
    summary.index = summary.index.str.replace('_year', '', regex=False)
    display(summary.sort_values('МО со значением'))

,год (медиана),разных лет,МО со значением
жилье_благоустр,2025.0,2,1326
детсады_охват,2024.0,3,1336
дороги_плохие_доля,2025.0,4,1346
доля_своих_доходов,2025.0,3,1352
расходы_на_омсу,2024.0,4,1352
жилье_на_жителя,2024.0,3,1373
жилье_ввод_на_жителя,2024.0,4,1387
средства_размещения,2025.0,4,1480
улицы,2024.0,3,1486
зарплата_крупные,2025.0,3,1488


## 4. Пространственные признаки

Считаются по границам из раздела 1 и численности населения. Справочник крупных городов не нужен: это те же МО, у которых население выше порога, поэтому признаки воспроизводятся на любом наборе МО.

Что получается: площадь и компактность территории, плотность населения, расстояние до ближайшего города от 100 и от 500 тысяч жителей, признак пригорода, расстояние до центра региона, число соседей по границе и доступность рынка (взвешенная по расстоянию сумма населения вокруг).

Центр региона определяется как самое населённое МО региона. Для большинства регионов это административный центр, но не для всех.

In [11]:
population = rosstat['население_среднегод'] if 'население_среднегод' in rosstat else None
geo_features = geo.load(mo_boundaries, base, population=population, oktmo_col='oktmo')
print(f'Географических признаков: {geo_features.shape[1]}')
display(geo_features.describe().T[['count', 'mean', '50%', 'max']])

МО с населением от 100 000: 216
МО с населением от 500 000: 31
Географических признаков: 14


,count,mean,50%,max
area_km2,1855.0,8016.662819,1865.346929,845938.790351
centroid_lat,1855.0,55.326174,55.614029,72.469054
centroid_lon,1855.0,59.308729,48.123586,177.459244
compactness,1855.0,0.224101,0.208187,0.77647
neighbors,1855.0,4.486253,5.0,17.0
population,1531.0,62459.784237,23488.0,1635559.0
density_per_km2,1531.0,1634.207485,12.409041,29442.518796
dist_to_city_100k_km,1855.0,138.471265,88.662602,2017.123255
dist_to_city_500k_km,1855.0,272.277395,155.955024,3432.552433
is_suburb,1531.0,0.233181,0.0,1.0


## 5. Сборка таблицы

Соединяем всё по ОКТМО. `common.attach` печатает покрытие каждого блока: для скольких МО признаки нашлись.

In [12]:
features = base.copy()
features = common.attach(features, rosstat_features, prefix='rs_')
features = common.attach(features, geo_features, prefix='geo_')

features = features.set_index('oktmo')
print(f'\nИтого: {len(features)} МО, {features.shape[1]} колонок')

rs_: 63 колонок, найдены для 1572 из 1855 МО (85%)
geo_: 14 колонок, найдены для 1855 из 1855 МО (100%)

Итого: 1855 МО, 80 колонок


### Проверка полноты

Признак, который есть лишь у части МО, в кластеризации создаёт проблемы: либо приходится выбрасывать МО, либо заполнять пропуски и получать искусственное сходство. Отдельно смотрим на признаки с малым покрытием и на МО, по которым почти ничего не собралось.

У Росстата пропуски часто означают не ноль, а скрытие данных по конфиденциальности (мало организаций в МО). Заполнять такие пропуски нулями нельзя.

In [13]:
numeric = features.select_dtypes('number')
coverage = numeric.notna().mean().sort_values()
print('Признаки с наименьшим покрытием:')
display(coverage.head(15).to_frame('доля МО со значением'))

by_mo = numeric.notna().mean(axis=1)
print(f'Медианная заполненность по МО: {by_mo.median():.0%}')
print('МО с наименьшей заполненностью:')
display(pd.DataFrame({'мо': features['mo'], 'заполнено': by_mo}).sort_values('заполнено').head(10))

Признаки с наименьшим покрытием:


,доля МО со значением
rs_жилье_благоустр,0.714825
rs_детсады_охват,0.720216
rs_дороги_плохие_доля,0.725606
rs_доля_своих_доходов,0.728841
rs_расходы_на_омсу,0.728841
rs_жилье_на_жителя,0.740162
rs_жилье_ввод_на_жителя,0.747709
rs_средства_размещения_на_1000,0.775741
rs_улицы_на_1000,0.778976
rs_лпу_на_1000,0.780054


Медианная заполненность по МО: 100%
МО с наименьшей заполненностью:


,мо,заполнено
oktmo,,
70630000,Куркинский муниципальный район,0.142857
17630000,Киржачский муниципальный район,0.142857
04620000,Казачинский муниципальный район,0.142857
76632000,Оловяннинский муниципальный район,0.142857
04649000,Северо-Енисейский муниципальный район,0.142857
58654000,Себежский муниципальный район,0.142857
29636000,Сухиничский муниципальный район,0.142857
29638000,Тарусский муниципальный район,0.142857
75659000,Чесменский муниципальный район,0.142857


In [14]:
features.to_csv(OUT / 'mo_features.csv')
print('Сохранено:', OUT / 'mo_features.csv')

Сохранено: results/mo_features.csv


In [ ]:
# Для кластеризации: признаки по ОКТМО, ключ тот же, что в базовом списке
static = pd.read_csv(OUT / 'mo_features.csv', dtype={'oktmo': str}).set_index('oktmo')

# Для анализа трат по месяцам: статичные признаки в каждой строке временного ряда
costs_keyed = costs.copy()
costs_keyed['oktmo'] = costs_keyed['oktmo'].map(common.normalize_oktmo)
costs_with_features = costs_keyed.merge(static.drop(columns=['mo']), left_on='oktmo',
                                        right_index=True, how='left')
print(f'Строк в таблице трат с признаками: {len(costs_with_features)}, колонок: {costs_with_features.shape[1]}')
display(costs_with_features.head(3))